# 04 — Hypothesis Testing (Phase 2)

For each hypothesis:

- **H0 / H1** are stated explicitly.
- The **test** is justified and its **assumptions** are checked.
- We report the test statistic, p-value, **effect size**, and a bootstrap 95% CI
  of the mean difference where applicable.
- Every interpretation carries an explicit **non-causation** caveat.

**Scope:** train + validation only. The held-out test set is untouched.

**Large-sample handling:** H3 and H4 use `MAX_N_PER_SAMPLE = 200_000` per group,
so the notebook never freezes on the ~27M-row rejected file. The reported `n`
always matches what was actually tested.

**Hypotheses tested:**

- H1 — DTI and default class
- H2 — Purpose and default class
- H3 — Risk_Score across state funded-proportion groups (rejected file)
- H4 — Requested amount: accepted vs rejected (subsampled)
- H5 — FICO and default class
- H6 — Revolving utilization and default class (optional, extra)

In [1]:
import sys, json, logging
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.splits import ChronologicalSplitter
from src.hypothesis import HypothesisTester

logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
OUT_TABLES = PROJECT_ROOT / 'outputs' / 'tables'; OUT_TABLES.mkdir(parents=True, exist_ok=True)

# Reproducibility + performance knobs (single source of truth)
SEED = 42
MAX_N_PER_SAMPLE = 200_000
tester = HypothesisTester(seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE)
PROJECT_ROOT

PosixPath('/Users/goncalo.ramos/Mestrado/CD/projeto/Project_Ciencias_Dados')

## 1. Load cleaned data and reuse the frozen split

In [2]:
with open(PROJECT_ROOT / 'outputs' / 'split_definition.json') as f:
    split_cfg = json.load(f)
splitter = ChronologicalSplitter(**split_cfg)
acc_clf = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_classification.parquet',
                          columns=['issue_d','target_class','dti','purpose','fico_range_low','revol_util'])
clf_eda = splitter.development(acc_clf)
del acc_clf
# All funded loans, including Current and other unknown outcomes, for cross-file comparisons.
funded = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'accepted_regression.parquet',
                         columns=['issue_d','loan_amnt','addr_state'])
funded_dev = splitter.development(funded)
del funded
rej = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'rejected_clean.parquet',
                       columns=['Application Date','Risk_Score','Amount Requested','State'])
rej_dev = splitter.development(rej, date_col='Application Date')
del rej
scope_rows = []
for name, frame, date_col in [('classification', clf_eda, 'issue_d'),
                               ('funded', funded_dev, 'issue_d'),
                               ('rejected', rej_dev, 'Application Date')]:
    assert frame[date_col].dt.to_period('M').le(pd.Period(split_cfg['val_end'], freq='M')).all()
    scope_rows.append({'cohort': name, 'n': len(frame), 'min_date': frame[date_col].min(),
                      'max_date': frame[date_col].max(), 'cutoff': split_cfg['val_end']})
pd.DataFrame(scope_rows).to_csv(OUT_TABLES / 'hypothesis_development_scope.csv', index=False)
pd.DataFrame(scope_rows)

,cohort,n,min_date,max_date,cutoff
0,classification,1129813,2007-06-01,2016-12-01,2016-12
1,funded,1321847,2007-06-01,2016-12-01,2016-12
2,rejected,11032222,2007-05-26,2016-12-31,2016-12


## 2. Assumption checks (normality)

Shapiro-Wilk is unstable for very large n, so we run it on a reproducible 5k
subsample. Rejection of normality justifies non-parametric location tests.

In [3]:
norm_rows = []
for col in ['dti', 'annual_inc', 'fico_range_low', 'loan_amnt']:
    if col in clf_eda.columns:
        r = tester.normality_report(clf_eda[col].dropna().values, seed=SEED)
        r['column'] = col
        norm_rows.append(r)
norm_df = pd.DataFrame(norm_rows).set_index('column')
norm_df.to_csv(OUT_TABLES / 'hypothesis_normality_checks.csv')
norm_df.round(4)

,shapiro_stat,shapiro_p,skew,n_used
column,,,,
dti,0.8726,0.0,4.4284,5000
fico_range_low,0.8881,0.0,1.3431,5000


## H1 — Debt-to-income ratio and default risk

- **H0:** the distribution of DTI is identical for `Default` (class 2) and `Good` (class 0) loans.
- **H1:** the distribution differs (two-sided).
- **Test:** Mann-Whitney U (DTI is skewed; Shapiro rejects normality).
- **Effect size:** Cliff's delta.
- **Expected direction:** defaulters should have higher DTI.

In [4]:
res_h1 = None
if {'dti', 'target_class'}.issubset(clf_eda.columns):
    a = clf_eda.loc[clf_eda['target_class'] == 2, 'dti'].dropna().values
    b = clf_eda.loc[clf_eda['target_class'] == 0, 'dti'].dropna().values
    print(f'H1 group sizes: Default={len(a):,}, Good={len(b):,}')
    res_h1 = tester.mannwhitney_test(
        a, b,
        name='H1 DTI Default vs Good',
        h0='DTI distribution identical across Default and Good',
        h1='DTI distribution differs across Default and Good (two-sided)',
        note='Observational association; not causal.',
        seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE,
    )
    if res_h1 is not None:
        print(res_h1.summary())
        print(f'  mean DTI Default = {a.mean():.3f} | mean DTI Good = {b.mean():.3f}')
else:
    print('H1 skipped: dti or target_class missing from clf_eda')
res_h1.to_dict() if res_h1 else None

H1 group sizes: Default=220,538, Good=899,125


H1 DTI Default vs Good | Mann-Whitney U | stat=1.149e+11, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.1587 | n=1119663, 95% CI of mean diff = [2.2883, 2.3946]
  mean DTI Default = 20.082 | mean DTI Good = 17.715


{'name': 'H1 DTI Default vs Good',
 'h0': 'DTI distribution identical across Default and Good',
 'h1': 'DTI distribution differs across Default and Good (two-sided)',
 'test': 'Mann-Whitney U',
 'statistic': 114877077302.5,
 'p_value': 0.0,
 'effect_size': 0.15867028246283363,
 'effect_size_name': "Cliff's delta",
 'ci_low': 2.288309628749998,
 'ci_high': 2.39464694,
 'n': 1119663,
 'n_a': 220538,
 'n_b': 899125,
 'note': 'Observational association; not causal.'}

## H2 — Loan purpose and default risk

- **H0:** `purpose` and `target_class` are independent.
- **H1:** they are associated.
- **Test:** Chi-square on the contingency table.
- **Assumption:** expected cell counts ≥ 5. Purposes accounting for <1% of all
  applications are collapsed into a single `OTHER_RARE` row.
- **Effect size:** Cramér's V (negligible if < 0.10).

**Important:** with n ≈ 1.1M, chi-square will be significant even for a negligible
association. The **effect size**, not the p-value, drives the conclusion.

In [5]:
res_h2 = None
if {'purpose', 'target_class'}.issubset(clf_eda.columns):
    table = pd.crosstab(clf_eda['purpose'], clf_eda['target_class'])
    # Collapse rare purposes (<1% of all applications)
    rare = table.sum(axis=1) < 0.01 * table.values.sum()
    if rare.any():
        other_row = table.loc[rare].sum(axis=0).rename('OTHER_RARE').to_frame().T
        table = pd.concat([table.loc[~rare], other_row])
    res_h2 = tester.chi_square_test(
        table,
        name='H2 Purpose vs default class',
        h0='purpose and target_class are independent',
        h1='purpose and target_class are associated',
        note='Observational association; not causal.',
    )
    table.to_csv(OUT_TABLES / 'hypothesis_purpose_vs_class_table.csv')
    if res_h2 is not None:
        print(res_h2.summary())
else:
    print('H2 skipped: purpose or target_class missing from clf_eda')
res_h2.to_dict() if res_h2 else None

[INFO] H2 Purpose vs default class: Cramér's V = 0.0384 is negligible (p is small because n = 1,129,813).
H2 Purpose vs default class | Chi-square | stat=3337, p=0.000e+00 -> REJECT H0 | Cramér's V=0.0384 | n=1129813


{'name': 'H2 Purpose vs default class',
 'h0': 'purpose and target_class are independent',
 'h1': 'purpose and target_class are associated',
 'test': 'Chi-square',
 'statistic': 3337.167673134212,
 'p_value': 0.0,
 'effect_size': 0.0384300307851398,
 'effect_size_name': "Cramér's V",
 'ci_low': None,
 'ci_high': None,
 'n': 1129813,
 'n_a': None,
 'n_b': None,
 'note': 'Observational association; not causal.'}

## H3 — Risk_Score across state funded-proportion groups (rejected file)

- **H0:** the distribution of `Risk_Score` (rejected file) is identical across
  states with high vs low development-period dataset funded proportions.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U. **Subsampled to ≤200,000 per group.**
- **Proxy note:** the state grouping is derived from the accepted file; the
  tested outcome (`Risk_Score`) comes entirely from the rejected file.

This cell prints **diagnostics** first, so if the grouping cannot be built you
will see exactly why (missing column, empty join, or empty groups).

The grouping uses all funded records in the development period, regardless of
repayment-label availability, and development-period rejected applications.
This is a dataset funded proportion, not a calibrated approval probability:
funding month differs from application date and repeated/withdrawn applications
cannot be reconciled across files. This proxy does not directly test whether
lower-score individual applicants are more likely to be rejected.

In [6]:
res_h3 = None

have_cols = (
    'Risk_Score' in rej_dev.columns
    and 'State' in rej_dev.columns
    and 'addr_state' in funded_dev.columns
)
print('H3 required columns present?', have_cols)

if have_cols:
    # Normalise state keys to upper-case, stripped strings
    rej_state_key = rej_dev['State'].astype(str).str.upper().str.strip()
    acc_state_key = funded_dev['addr_state'].astype(str).str.upper().str.strip()

    acc_state = acc_state_key.value_counts().rename('n_accepted')
    rej_state = rej_state_key.value_counts().rename('n_rejected')
    joined = pd.concat([acc_state, rej_state], axis=1).dropna()
    joined['funded_proportion'] = joined['n_accepted'] / (joined['n_accepted'] + joined['n_rejected'])

    print('joined states:', len(joined))
    print(joined.head())

    if len(joined) >= 2:
        median_ratio = joined['funded_proportion'].median()
        high_states = joined.index[joined['funded_proportion'] >= median_ratio]
        low_states  = joined.index[joined['funded_proportion'] <  median_ratio]
        print('high_states:', list(high_states))
        print('low_states :', list(low_states))

        rng = np.random.default_rng(SEED)
        a_full = rej_dev.loc[rej_state_key.isin(high_states), 'Risk_Score'].dropna().values
        b_full = rej_dev.loc[rej_state_key.isin(low_states),  'Risk_Score'].dropna().values
        print(f'H3 full group sizes: high={len(a_full):,}, low={len(b_full):,}')

        a = a_full if len(a_full) <= MAX_N_PER_SAMPLE else rng.choice(a_full, MAX_N_PER_SAMPLE, replace=False)
        b = b_full if len(b_full) <= MAX_N_PER_SAMPLE else rng.choice(b_full, MAX_N_PER_SAMPLE, replace=False)
        print(f'H3 sampled group sizes: high={len(a):,}, low={len(b):,}')

        res_h3 = tester.mannwhitney_test(
            a, b,
            name='H3 Risk_Score High vs Low funded-proportion states (rejected file)',
            h0='Risk_Score distribution identical across state groups',
            h1='Risk_Score distribution differs across state groups (two-sided)',
            note=(f'Reproducibly subsampled to <= {MAX_N_PER_SAMPLE:,} per group; '
                  f'state grouping derived from historical dataset funded proportions; observational.'),
            seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE,
        )
        joined.to_csv(OUT_TABLES / 'hypothesis_state_approval_ratio.csv')
        if res_h3 is not None:
            print(res_h3.summary())
            print(f'  median Risk_Score high = {np.median(a):.2f} | '
                  f'median Risk_Score low = {np.median(b):.2f}')
    else:
        print('H3 skipped: state join produced fewer than 2 states.')
else:
    print('H3 skipped: a required column is missing.')

res_h3.to_dict() if res_h3 else None

H3 required columns present? True


joined states: 51
    n_accepted  n_rejected  funded_proportion
CA      187415     1350307           0.121878
NY      109594      847404           0.114519
TX      108175      974797           0.099887
FL       92668      845216           0.098805
IL       53192      405099           0.116066


high_states: ['CA', 'NY', 'IL', 'NJ', 'VA', 'MI', 'MD', 'AZ', 'MA', 'WA', 'CO', 'MN', 'CT', 'NV', 'WI', 'OR', 'KS', 'UT', 'NH', 'RI', 'MT', 'DC', 'AK', 'WY', 'VT', 'ND']
low_states : ['TX', 'FL', 'PA', 'OH', 'GA', 'NC', 'IN', 'MO', 'TN', 'AL', 'SC', 'LA', 'KY', 'OK', 'AR', 'NM', 'MS', 'HI', 'WV', 'DE', 'NE', 'SD', 'ME', 'ID', 'IA']


H3 full group sizes: high=2,404,783, low=2,177,112
H3 sampled group sizes: high=200,000, low=200,000


H3 Risk_Score High vs Low funded-proportion states (rejected file) | Mann-Whitney U | stat=2.104e+10, p=1.393e-179 -> REJECT H0 | Cliff's delta=0.0522 | n=400000, 95% CI of mean diff = [5.8312, 6.6510]
  median Risk_Score high = 644.00 | median Risk_Score low = 638.00


{'name': 'H3 Risk_Score High vs Low funded-proportion states (rejected file)',
 'h0': 'Risk_Score distribution identical across state groups',
 'h1': 'Risk_Score distribution differs across state groups (two-sided)',
 'test': 'Mann-Whitney U',
 'statistic': 21043379247.0,
 'p_value': 1.3930386738487491e-179,
 'effect_size': 0.05216896235,
 'effect_size_name': "Cliff's delta",
 'ci_low': 5.831213125000005,
 'ci_high': 6.6510261250000156,
 'n': 400000,
 'n_a': 200000,
 'n_b': 200000,
 'note': 'Reproducibly subsampled to <= 200,000 per group; state grouping derived from historical dataset funded proportions; observational.'}

## H4 — Requested amount, accepted vs rejected

- **H0:** `loan_amnt` (accepted) and `Amount Requested` (rejected) have the same distribution.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U.
- **Design:** reproducible subsample of ≤200,000 rows per side, so the reported
  `n` matches the note and the notebook finishes in seconds.

Accepted-side amounts now cover all development-period funded loans, including
Current loans; they are not conditioned on a known repayment outcome. Funded
amount and rejected requested amount remain imperfectly comparable snapshots.

In [7]:
res_h4 = None

have_h4_cols = ('Amount Requested' in rej_dev.columns) and ('loan_amnt' in funded_dev.columns)
print('H4 required columns present?', have_h4_cols)

if have_h4_cols:
    rng = np.random.default_rng(SEED)
    a_full = funded_dev['loan_amnt'].dropna().values
    b_full = rej_dev['Amount Requested'].dropna().values
    print(f'H4 full group sizes: accepted={len(a_full):,}, rejected={len(b_full):,}')

    a = a_full if len(a_full) <= MAX_N_PER_SAMPLE else rng.choice(a_full, MAX_N_PER_SAMPLE, replace=False)
    b = b_full if len(b_full) <= MAX_N_PER_SAMPLE else rng.choice(b_full, MAX_N_PER_SAMPLE, replace=False)
    print(f'H4 sampled group sizes: accepted={len(a):,}, rejected={len(b):,}')

    res_h4 = tester.mannwhitney_test(
        a, b,
        name='H4 Requested amount Accepted vs Rejected',
        h0='Requested amount distribution identical across groups',
        h1='Requested amount distribution differs across groups (two-sided)',
        note=(f'Reproducibly subsampled to <= {MAX_N_PER_SAMPLE:,} per group; '
              f'observational; not causal. Accepted uses funded loan_amnt.'),
        seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE,
    )
    if res_h4 is not None:
        print(res_h4.summary())
        print(f'  median accepted = {np.median(a):.0f} | median rejected = {np.median(b):.0f}')
else:
    print('H4 skipped: a required column is missing.')

res_h4.to_dict() if res_h4 else None

H4 required columns present? True
H4 full group sizes: accepted=1,321,847, rejected=11,032,222
H4 sampled group sizes: accepted=200,000, rejected=200,000


H4 Requested amount Accepted vs Rejected | Mann-Whitney U | stat=2.417e+10, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.2084 | n=400000, 95% CI of mean diff = [1323.7381, 1482.3820]
  median accepted = 12925 | median rejected = 10000


{'name': 'H4 Requested amount Accepted vs Rejected',
 'h0': 'Requested amount distribution identical across groups',
 'h1': 'Requested amount distribution differs across groups (two-sided)',
 'test': 'Mann-Whitney U',
 'statistic': 24167240815.0,
 'p_value': 0.0,
 'effect_size': 0.20836204075,
 'effect_size_name': "Cliff's delta",
 'ci_low': 1323.738076981248,
 'ci_high': 1482.3819982375,
 'n': 400000,
 'n_a': 200000,
 'n_b': 200000,
 'note': 'Reproducibly subsampled to <= 200,000 per group; observational; not causal. Accepted uses funded loan_amnt.'}

## H5 — FICO and default class

- **H0:** the distribution of `fico_range_low` is identical for `Default` and `Good` loans.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U (FICO is right-skewed; Shapiro rejects normality).
- **Effect size:** Cliff's delta.
- **Expected direction:** defaulters should have **lower** FICO.

This is the substantive replacement for the earlier 36m-vs-60m parametric check.

In [8]:
res_h5 = None

have_h5_cols = {'fico_range_low', 'target_class'}.issubset(clf_eda.columns)
print('H5 required columns present?', have_h5_cols)

if have_h5_cols:
    a = clf_eda.loc[clf_eda['target_class'] == 2, 'fico_range_low'].dropna().values
    b = clf_eda.loc[clf_eda['target_class'] == 0, 'fico_range_low'].dropna().values
    print(f'H5 group sizes: Default={len(a):,}, Good={len(b):,}')

    res_h5 = tester.mannwhitney_test(
        a, b,
        name='H5 FICO Default vs Good',
        h0='FICO distribution identical across Default and Good',
        h1='FICO distribution differs across Default and Good (two-sided)',
        note='Observational association; not causal.',
        seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE,
    )
    if res_h5 is not None:
        print(res_h5.summary())
        print(f'  mean FICO Default = {a.mean():.2f} | mean FICO Good = {b.mean():.2f}')
else:
    print('H5 skipped: fico_range_low or target_class missing from clf_eda')

res_h5.to_dict() if res_h5 else None

H5 required columns present? True
H5 group sizes: Default=220,556, Good=899,155


H5 FICO Default vs Good | Mann-Whitney U | stat=8.073e+10, p=0.000e+00 -> REJECT H0 | Cliff's delta=-0.1858 | n=1119711, 95% CI of mean diff = [-10.4058, -10.0464]
  mean FICO Default = 686.95 | mean FICO Good = 697.09


{'name': 'H5 FICO Default vs Good',
 'h0': 'FICO distribution identical across Default and Good',
 'h1': 'FICO distribution differs across Default and Good (two-sided)',
 'test': 'Mann-Whitney U',
 'statistic': 80733134547.5,
 'p_value': 0.0,
 'effect_size': -0.18580511450226228,
 'effect_size_name': "Cliff's delta",
 'ci_low': -10.40581249999995,
 'ci_high': -10.04637499999995,
 'n': 1119711,
 'n_a': 220556,
 'n_b': 899155,
 'note': 'Observational association; not causal.'}

## H6 (optional) — Revolving utilization and default class

- **H0:** the distribution of `revol_util` is identical for `Default` and `Good` loans.
- **H1:** the distributions differ (two-sided).
- **Test:** Mann-Whitney U.
- **Effect size:** Cliff's delta.

In [9]:
res_h6 = None

have_h6_cols = {'revol_util', 'target_class'}.issubset(clf_eda.columns)
print('H6 required columns present?', have_h6_cols)

if have_h6_cols:
    a = clf_eda.loc[clf_eda['target_class'] == 2, 'revol_util'].dropna().values
    b = clf_eda.loc[clf_eda['target_class'] == 0, 'revol_util'].dropna().values
    print(f'H6 group sizes: Default={len(a):,}, Good={len(b):,}')

    res_h6 = tester.mannwhitney_test(
        a, b,
        name='H6 Revolving utilization Default vs Good',
        h0='revol_util distribution identical across Default and Good',
        h1='revol_util distribution differs across Default and Good (two-sided)',
        note='Observational association; not causal.',
        seed=SEED, max_n_per_sample=MAX_N_PER_SAMPLE,
    )
    if res_h6 is not None:
        print(res_h6.summary())
        print(f'  mean revol_util Default = {a.mean():.2f} | mean revol_util Good = {b.mean():.2f}')
else:
    print('H6 skipped: revol_util or target_class missing from clf_eda')

res_h6.to_dict() if res_h6 else None

H6 required columns present? True
H6 group sizes: Default=220,428, Good=898,651


H6 Revolving utilization Default vs Good | Mann-Whitney U | stat=1.066e+11, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.0765 | n=1119079, 95% CI of mean diff = [3.0831, 3.3701]
  mean revol_util Default = 55.94 | mean revol_util Good = 52.70


{'name': 'H6 Revolving utilization Default vs Good',
 'h0': 'revol_util distribution identical across Default and Good',
 'h1': 'revol_util distribution differs across Default and Good (two-sided)',
 'test': 'Mann-Whitney U',
 'statistic': 106621155096.0,
 'p_value': 0.0,
 'effect_size': 0.07650377409813788,
 'effect_size_name': "Cliff's delta",
 'ci_low': 3.083136966250003,
 'ci_high': 3.370143968749996,
 'n': 1119079,
 'n_a': 220428,
 'n_b': 898651,
 'note': 'Observational association; not causal.'}

## Consolidated results

One row per hypothesis. Any `None` input triggers a `[MISSING]` warning, so
silent failures are impossible.

In [10]:
all_results = [res_h1, res_h2, res_h3, res_h4, res_h5, res_h6]
results_df = tester.results_to_dataframe(all_results)
results_df.to_csv(OUT_TABLES / 'hypothesis_results.csv', index=False)

print()
print('=== Summary ===')
for r in [res_h1, res_h2, res_h3, res_h4, res_h5, res_h6]:
    if r is not None:
        print(r.summary())

print()
print('Number of successful tests:', len(results_df))
results_df[['name','test','statistic','p_value','effect_size',
            'effect_size_name','n','n_a','n_b','ci_low','ci_high']]


=== Summary ===
H1 DTI Default vs Good | Mann-Whitney U | stat=1.149e+11, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.1587 | n=1119663, 95% CI of mean diff = [2.2883, 2.3946]
H2 Purpose vs default class | Chi-square | stat=3337, p=0.000e+00 -> REJECT H0 | Cramér's V=0.0384 | n=1129813
H3 Risk_Score High vs Low funded-proportion states (rejected file) | Mann-Whitney U | stat=2.104e+10, p=1.393e-179 -> REJECT H0 | Cliff's delta=0.0522 | n=400000, 95% CI of mean diff = [5.8312, 6.6510]
H4 Requested amount Accepted vs Rejected | Mann-Whitney U | stat=2.417e+10, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.2084 | n=400000, 95% CI of mean diff = [1323.7381, 1482.3820]
H5 FICO Default vs Good | Mann-Whitney U | stat=8.073e+10, p=0.000e+00 -> REJECT H0 | Cliff's delta=-0.1858 | n=1119711, 95% CI of mean diff = [-10.4058, -10.0464]
H6 Revolving utilization Default vs Good | Mann-Whitney U | stat=1.066e+11, p=0.000e+00 -> REJECT H0 | Cliff's delta=0.0765 | n=1119079, 95% CI of mean diff = [3.0831,

,name,test,statistic,p_value,effect_size,effect_size_name,n,n_a,n_b,ci_low,ci_high
0,H1 DTI Default vs Good,Mann-Whitney U,1.148771e+11,0.000000e+00,0.158670,Cliff's delta,1119663,220538.0,899125.0,2.288310,2.394647
1,H2 Purpose vs default class,Chi-square,3.337168e+03,0.000000e+00,0.038430,Cramér's V,1129813,NaN,NaN,NaN,NaN
2,H3 Risk_Score High vs Low funded-proportion st...,Mann-Whitney U,2.104338e+10,1.393039e-179,0.052169,Cliff's delta,400000,200000.0,200000.0,5.831213,6.651026
3,H4 Requested amount Accepted vs Rejected,Mann-Whitney U,2.416724e+10,0.000000e+00,0.208362,Cliff's delta,400000,200000.0,200000.0,1323.738077,1482.381998
4,H5 FICO Default vs Good,Mann-Whitney U,8.073313e+10,0.000000e+00,-0.185805,Cliff's delta,1119711,220556.0,899155.0,-10.405812,-10.046375
5,H6 Revolving utilization Default vs Good,Mann-Whitney U,1.066212e+11,0.000000e+00,0.076504,Cliff's delta,1119079,220428.0,898651.0,3.083137,3.370144


## Interpretation and limitations

The following cell reports direction, effect size and mean-difference CI from the
current run. These are observational, unadjusted comparisons, not causal effects.
Mann–Whitney tests distributions/rank ordering; the bootstrap CI describes a mean
difference, not the U statistic. CIs use reproducible samples capped at 200,000
per group, whereas H1/H5/H6 rank tests can use the full development groups.
Numerically underflowed p-values printed as zero mean extremely small values.

Cliff's delta thresholds used descriptively are <0.147 negligible, <0.33 small,
<0.474 medium, otherwise large; conventions are not universal. Large samples can
make negligible associations statistically significant. H3 is a state-level proxy,
H4 compares different snapshots, and neither identifies individual approval effects.
State and vintage composition, repeated borrowers and unequal follow-up remain
uncontrolled. No verified common label-observation cutoff is available.

In [11]:
for r in all_results:
    if r is None:
        continue
    if r.effect_size_name == "Cliff's delta":
        magnitude = abs(r.effect_size)
        label = ('negligible' if magnitude < 0.147 else 'small' if magnitude < 0.33
                 else 'medium' if magnitude < 0.474 else 'large')
        direction = 'higher in first group' if r.effect_size > 0 else 'lower in first group'
        print(f'{r.name}: {direction}; {label} rank effect ({r.effect_size:.4f}); mean-difference CI [{r.ci_low:.3f}, {r.ci_high:.3f}].')
    else:
        print(f'{r.name}: Cramér V={r.effect_size:.4f}; interpret practical size separately from significance.')


H1 DTI Default vs Good: higher in first group; small rank effect (0.1587); mean-difference CI [2.288, 2.395].
H2 Purpose vs default class: Cramér V=0.0384; interpret practical size separately from significance.
H3 Risk_Score High vs Low funded-proportion states (rejected file): higher in first group; negligible rank effect (0.0522); mean-difference CI [5.831, 6.651].
H4 Requested amount Accepted vs Rejected: higher in first group; small rank effect (0.2084); mean-difference CI [1323.738, 1482.382].
H5 FICO Default vs Good: lower in first group; small rank effect (-0.1858); mean-difference CI [-10.406, -10.046].
H6 Revolving utilization Default vs Good: higher in first group; negligible rank effect (0.0765); mean-difference CI [3.083, 3.370].
